[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C28_Frontier_Diffusion_Course/01_latent_diffusion/01_latent_diffusion.ipynb)

# 01 · Latent Diffusion（用 numpy 玩具复现）

目标：用纯 numpy 把 **Stable Diffusion 的核心——latent 扩散**从零跑通：① 玩具 VAE 把高维「图」压到低维 latent；② 在 latent 上做闭式加噪与 DDPM 采样；③ 解码回「图」并验证落在数据流形附近。

路线：玩具图像数据 → 玩具 VAE(编/解码 + 压缩比) → latent 缩放的必要性 → latent 上闭式加噪 → 闭式最优去噪 → DDPM 采样 → ✏️ 练习 → 📖 答案 → 🧪 真实 SD 配置胶囊。

> 心智模型：**VAE = 学过的有损压缩；latent 扩散 = 在压缩域里做和像素扩散一模一样的加噪/去噪**。维度小到能 print，但逻辑与真实 SD 一一对应。

## 1 · 玩具「图像」：活在低维子空间里的高维向量

真实图像虽是百万维，但有大量冗余——它们其实活在一个低维流形附近（这正是能压缩的原因）。

我们造一批 `D_pix=48` 维的「图」，但它们由 `d_lat=4` 个隐因子线性生成（再加一点噪声）。所以理论上 4 维 latent 就能近乎无损表示它们——**有冗余可压**。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

D_pix = 48          # 「像素」维度（玩具图）
d_lat = 4           # 真实隐因子数（latent 维度）
n = 3000

# 真实生成过程：z_true (n,4) --随机线性映射 W (4,48)--> 像素 + 小噪声
W_true = rng.standard_normal((d_lat, D_pix))
z_true = rng.standard_normal((n, d_lat))
# 给隐因子不同尺度，模拟「有的语义方向更重要」
z_true *= np.array([3.0, 2.0, 1.0, 0.5])
X = z_true @ W_true + 0.05 * rng.standard_normal((n, D_pix))
X = X - X.mean(0)   # 中心化（VAE 前的常规预处理）

print('玩具图 X shape:', X.shape, ' 每张图', D_pix, '维')
# 用 SVD 看有效维度：前 4 个奇异值应当主导（因为数据本质 4 维）
s = np.linalg.svd(X, compute_uv=False)
energy = s**2 / (s**2).sum()
print('奇异值能量占比 前6:', np.round(energy[:6], 3))
print(f'前 {d_lat} 维累计能量: {energy[:d_lat].sum():.3f}')
assert energy[:d_lat].sum() > 0.98, '数据应当几乎完全活在 4 维子空间'
print('✅ 48 维的图其实活在 ~4 维子空间 -> 有冗余可压，这正是 latent 扩散的前提')

## 2 · 玩具 VAE：用 PCA 当编码器/解码器

真实 VAE 是深度卷积网络。这里用**线性 VAE**（PCA）抓住本质：编码器 `E` 把 `D_pix` 维投到 `d_lat` 维，解码器 `D` 投回去。

PCA 的编码矩阵 `Wenc`（取前 `d_lat` 个右奇异向量）满足 `Wenc @ Wenc.T = I`，解码就是 `Wenc.T`。验证 **`D(E(x)) ≈ x`（重建）** 且 **latent 维度远小于像素（压缩）**。

In [ ]:
# 用训练集 X 拟合 PCA 作为 VAE
U, S, Vt = np.linalg.svd(X - X.mean(0), full_matrices=False)
Wenc = Vt[:d_lat]                 # (d_lat, D_pix)，正交行
x_mean = X.mean(0)

def vae_encode(x):
    '''E: 像素 -> latent。'''
    return (x - x_mean) @ Wenc.T   # (.., d_lat)

def vae_decode(z):
    '''D: latent -> 像素。'''
    return z @ Wenc + x_mean       # (.., D_pix)

# 检验正交性与重建
assert np.allclose(Wenc @ Wenc.T, np.eye(d_lat), atol=1e-8), '编码矩阵应行正交'
Z = vae_encode(X)
X_rec = vae_decode(Z)
rec_err = np.mean((X - X_rec)**2)
var_x = np.mean((X - x_mean)**2)
print(f'latent Z shape: {Z.shape}  压缩比 = {D_pix/d_lat:.0f}x')
print(f'重建 MSE = {rec_err:.5f}  (数据方差 {var_x:.3f})  相对误差 {rec_err/var_x:.4f}')
assert rec_err / var_x < 0.02, 'VAE 往返应近似恒等（重建误差<2%）'
print('✅ VAE 往返 D(E(x))≈x：12x 压缩下重建误差<2% —— 感知压缩成立')

## 3 · latent 缩放：为什么 SD 要乘 0.18215

扩散假设加的是**单位方差**噪声 `ε~N(0,I)`。但 VAE 输出的 latent 方差未必是 1（我们的 latent 各通道方差由隐因子尺度决定，差异很大）。

若不缩放，各通道 SNR 严重不均、扩散训练不稳。SD 的做法：乘一个标量 `scale` 让 latent 整体方差≈1。我们算出这个 scale 并验证。

In [ ]:
# VAE 输出 latent 的各通道方差（很不均匀 -> 这就是问题）
ch_var = Z.var(0)
print('latent 各通道方差:', np.round(ch_var, 3))
print('整体 std:', Z.std())

# SD 的缩放：用一个标量让整体 std≈1（SD 实测得 0.18215）
scale = 1.0 / Z.std()
Z_scaled = Z * scale
print(f'\n缩放系数 scale = {scale:.4f}')
print(f'缩放后整体 std = {Z_scaled.std():.4f} (目标≈1)')
assert abs(Z_scaled.std() - 1.0) < 1e-6, '缩放后整体方差应≈1'
# 流程约定：编码后乘 scale，解码前除 scale
def encode_for_diffusion(x):
    return vae_encode(x) * scale
def decode_from_diffusion(z):
    return vae_decode(z / scale)
assert np.allclose(decode_from_diffusion(encode_for_diffusion(X)), X_rec, atol=1e-8)
print('✅ latent 缩放使整体方差≈1，匹配扩散的单位噪声假设（SD 的 0.18215 同理）')

## 4 · 在 latent 上闭式加噪：和像素扩散同一公式

关键论点：**latent 扩散不改扩散数学，只换空间**。闭式加噪 `z_t = √ᾱ_t·z0 + √(1-ᾱ_t)·ε` 里把 `x` 换成 `z` 即可。

我们建噪声表，在 latent 上加噪，并验证加噪后 latent 的边际方差符合理论（确认 latent 已是单位方差、公式正确）。

In [ ]:
def make_schedule(T=200, beta_min=1e-4, beta_max=0.02):
    betas = np.linspace(beta_min, beta_max, T)
    abar = np.cumprod(1.0 - betas)
    return betas, abar

T = 200
betas, abar = make_schedule(T)

def q_sample(z0, t, eps, abar):
    a = abar[t]
    return np.sqrt(a) * z0 + np.sqrt(1 - a) * eps

Z0 = encode_for_diffusion(X)            # 单位方差 latent
for t in [0, 50, 120, 199]:
    eps = rng.standard_normal(Z0.shape)
    Zt = q_sample(Z0, t, eps, abar)
    # 理论方差: ᾱ·Var(z0) + (1-ᾱ)·1 ；z0 单位方差 -> 总方差≈1 全程
    th = abar[t] * Z0.var() + (1 - abar[t])
    print(f't={t:3d}  ᾱ={abar[t]:.3f}  经验Var={Zt.var():.3f}  理论={th:.3f}')
    assert abs(Zt.var() - th) < 0.05
print('✅ latent 加噪的边际方差全程≈1，与理论一致 —— 单位方差 latent 让扩散数值稳定')
print('   (这正是第 3 节缩放的回报：若不缩放，这里方差会偏离 1、SNR 失衡)')

## 5 · 玩具去噪器：闭式最优 ε 预测

真实 SD 用大网络学 `εθ(z_t,t)`。玩具里我们的 latent 近似各向同性高斯 `N(0,I)`，此时**最优去噪有闭式解**，可当作「完美训练好的网络」来验证采样逻辑。

对 `z0~N(0,σ0²I)`、`z_t=√ᾱ·z0+√(1-ᾱ)·ε`，后验均值给出最优 `ε̂`。我们直接用这个闭式去噪器，专注验证**采样**这一步。

In [ ]:
# latent 近似 N(0, I)（已缩放）。对标准高斯先验，给定 z_t，
# 最优 x0 估计 = √ᾱ·z_t / (ᾱ + (1-ᾱ)) = √ᾱ·z_t  (因 z0 单位方差)，
# 对应最优 ε 估计由 z_t = √ᾱ·x0̂ + √(1-ᾱ)·ε̂ 反解。
def optimal_eps(z_t, t, abar, sigma0_sq=1.0):
    a = abar[t]
    # 后验 E[z0|z_t]（高斯先验 N(0,sigma0_sq I)）
    z0_hat = (np.sqrt(a) * sigma0_sq / (a * sigma0_sq + (1 - a))) * z_t
    eps_hat = (z_t - np.sqrt(a) * z0_hat) / np.sqrt(1 - a)
    return eps_hat, z0_hat

# 验证：用真实 (z0, eps) 造 z_t，闭式去噪器恢复的 z0_hat 应与真 z0 高度相关
z0 = Z0[:500]
t = 80
eps = rng.standard_normal(z0.shape)
z_t = q_sample(z0, t, eps, abar)
eps_hat, z0_hat = optimal_eps(z_t, t, abar)
corr = np.corrcoef(z0_hat.ravel(), z0.ravel())[0, 1]
print(f't={t}: 去噪恢复 z0 与真 z0 的相关系数 = {corr:.3f}')
assert corr > 0.5, '中等噪声下应能部分恢复 z0'
# t 越小（噪声越少）恢复越准
_, z0_hat_lo = optimal_eps(q_sample(z0, 5, eps, abar), 5, abar)
corr_lo = np.corrcoef(z0_hat_lo.ravel(), z0.ravel())[0, 1]
assert corr_lo > corr, '噪声越小恢复越准'
print(f't=5 (噪声更小): 相关系数 = {corr_lo:.3f} > {corr:.3f} ✅')
print('✅ 闭式最优去噪器就绪 —— 当作「完美网络」来验证 DDPM 采样')

## 6 · DDPM 采样：从 latent 噪声生成 + 解码

端到端：从 `z_T~N(0,I)` 出发，用 DDPM 反向公式逐步去噪到 `ẑ_0`，再 `decode` 回「图」。

DDPM 一步：`z_{t-1} = 1/√α_t · (z_t - (1-α_t)/√(1-ᾱ_t)·ε̂) + σ_t·noise`。验证：采样出的「图」落在数据流形附近（重建误差量级，而非随机）。

In [ ]:
alphas = 1.0 - betas

def ddpm_sample(n_samples, abar, betas, eps_fn, d=d_lat, seed=1):
    g = np.random.default_rng(seed)
    z = g.standard_normal((n_samples, d))     # z_T ~ N(0,I)
    T = len(betas)
    for t in range(T - 1, -1, -1):
        eps_hat, _ = eps_fn(z, t, abar)
        a_t = alphas[t]; ab_t = abar[t]
        mean = (z - (1 - a_t) / np.sqrt(1 - ab_t) * eps_hat) / np.sqrt(a_t)
        if t > 0:
            z = mean + np.sqrt(betas[t]) * g.standard_normal(z.shape)
        else:
            z = mean
    return z

z_gen = ddpm_sample(800, abar, betas, optimal_eps)
print('采样得到 latent:', z_gen.shape, ' std≈', round(z_gen.std(), 2))
# 解码回「图」
x_gen = decode_from_diffusion(z_gen)
# 检验：生成图应落在数据子空间内（被 VAE 重建几乎不变），而纯随机图不会
def reproj_err(x):
    return np.mean((x - vae_decode(vae_encode(x)))**2)
err_gen = reproj_err(x_gen)
err_rand = reproj_err(rng.standard_normal((800, D_pix)) * X.std())
print(f'生成图 的重投影误差 = {err_gen:.4f}')
print(f'纯随机图 的重投影误差 = {err_rand:.4f}')
assert err_gen < err_rand * 0.2, '生成图应落在数据流形(子空间)内，远优于随机'
print('✅ 端到端跑通：latent 噪声 -DDPM-> ẑ0 -解码-> 图，落在数据流形附近')

---
## ✏️ 练习 1：前向加噪（闭式）

实现 latent 空间的闭式前向加噪。给定 `z0`、时间步 `t`、噪声 `eps`、累积系数 `abar`，返回 `z_t`。

公式：$z_t=\sqrt{\bar\alpha_t}\,z_0+\sqrt{1-\bar\alpha_t}\,\epsilon$。

In [ ]:
def forward_noise(z0, t, eps, abar):
    # TODO: 实现闭式加噪 z_t = √ᾱ_t·z0 + √(1-ᾱ_t)·eps
    #       a = abar[t]; 注意逐元素 sqrt
    raise NotImplementedError
    return z_t

In [ ]:
# —— 练习 1 自测 ——
z0 = rng.standard_normal((100, d_lat))
eps = rng.standard_normal((100, d_lat))
# t=0 时 ᾱ≈1 -> z_t≈z0；t=T-1 时 ᾱ→小 -> z_t≈eps
zt0 = forward_noise(z0, 0, eps, abar)
ztT = forward_noise(z0, T - 1, eps, abar)
assert np.allclose(zt0, np.sqrt(abar[0]) * z0 + np.sqrt(1 - abar[0]) * eps)
# 与第 4 节的 q_sample 对拍
for tt in [10, 90, 150]:
    assert np.allclose(forward_noise(z0, tt, eps, abar), q_sample(z0, tt, eps, abar))
# 加噪后信号占比应随 t 单调下降
sig = [np.sqrt(abar[tt]) for tt in range(T)]
assert all(sig[i] >= sig[i+1] for i in range(T-1)), '信号系数应单调下降'
print('✅ 练习 1 通过：闭式前向加噪正确，与参考 q_sample 一致')

## ✏️ 练习 2：反向去噪一步（DDPM）

实现 DDPM 反向单步：给定 `z_t`、`t`、预测噪声 `eps_hat`、噪声表，返回 `z_{t-1}`（`t=0` 时无随机项）。

$z_{t-1}=\frac{1}{\sqrt{\alpha_t}}\Big(z_t-\frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\hat\epsilon\Big)+[t>0]\,\sqrt{\beta_t}\,n$

In [ ]:
def ddpm_step(z_t, t, eps_hat, alphas, abar, betas, noise):
    # TODO: 算后验均值 mean；若 t>0 加 √β_t·noise，否则返回 mean
    raise NotImplementedError
    return z_prev

In [ ]:
# —— 练习 2 自测 ——
zt = rng.standard_normal((50, d_lat))
eps_hat = rng.standard_normal((50, d_lat))
noise = rng.standard_normal((50, d_lat))
# t=0 应无随机项：两次调用结果相同（与传入 noise 无关）
z_a = ddpm_step(zt, 0, eps_hat, alphas, abar, betas, noise)
z_b = ddpm_step(zt, 0, eps_hat, alphas, abar, betas, rng.standard_normal((50, d_lat)))
assert np.allclose(z_a, z_b), 't=0 不应有随机项'
# t>0 时含随机项：不同 noise 给不同结果
z_c = ddpm_step(zt, 50, eps_hat, alphas, abar, betas, noise)
z_d = ddpm_step(zt, 50, eps_hat, alphas, abar, betas, noise + 1.0)
assert not np.allclose(z_c, z_d), 't>0 应含随机项'
# 均值部分核对
a_t, ab_t = alphas[50], abar[50]
mean_expect = (zt - (1 - a_t)/np.sqrt(1 - ab_t)*eps_hat)/np.sqrt(a_t)
assert np.allclose(z_c - np.sqrt(betas[50])*noise, mean_expect, atol=1e-8)
print('✅ 练习 2 通过：DDPM 反向单步正确（t=0 去随机、均值公式对）')

## ✏️ 练习 3：latent 编解码与压缩比

实现一个通用线性 VAE 构造器：给数据 `X` 和目标 latent 维 `k`，返回 `(encode, decode, ratio)`，其中 `ratio` 是压缩比 `D/k`。

要求：用 SVD 取前 `k` 个主方向；`encode/decode` 含去均值/加均值；验证重建误差随 `k` 增大而下降。

In [ ]:
def make_linear_vae(X, k):
    # TODO: 用 SVD 取前 k 个右奇异向量当编码矩阵 Wk (k, D)
    #       encode(x) = (x-mean)@Wk.T ; decode(z) = z@Wk + mean
    #       返回 (encode, decode, D/k)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
enc2, dec2, r2 = make_linear_vae(X, 2)
enc4, dec4, r4 = make_linear_vae(X, 4)
assert abs(r2 - D_pix/2) < 1e-9 and abs(r4 - D_pix/4) < 1e-9
err2 = np.mean((X - dec2(enc2(X)))**2)
err4 = np.mean((X - dec4(enc4(X)))**2)
print(f'k=2: 压缩{r2:.0f}x 重建MSE={err2:.4f}')
print(f'k=4: 压缩{r4:.0f}x 重建MSE={err4:.4f}')
assert err4 < err2, 'latent 维度越大重建越准（压缩-质量权衡）'
assert err4 < 0.01, 'k=4 应几乎无损（数据本质 4 维）'
# 编码维度正确
assert enc4(X).shape == (len(X), 4)
print('✅ 练习 3 通过：latent 维度↑ -> 压缩比↓、重建↑，体现感知压缩的核心权衡')

## ✏️ 练习 4：端到端采样并验证落在流形

组合前面的零件，写一个 `latent_pipeline(n_samples)`：DDPM 在 latent 采样 → 解码 → 返回生成「图」。

（用第 6 节的 `ddpm_sample` 与第 5 节的 `optimal_eps`、第 3 节的 `decode_from_diffusion`。）

In [ ]:
def latent_pipeline(n_samples, seed=7):
    # TODO: z_gen = ddpm_sample(n_samples, abar, betas, optimal_eps, seed=seed)
    #       返回 decode_from_diffusion(z_gen)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
x_out = latent_pipeline(600)
assert x_out.shape == (600, D_pix)
# 生成图应落在数据子空间（重投影误差小），统计接近真实数据
def reproj(x): return np.mean((x - vae_decode(vae_encode(x)))**2)
assert reproj(x_out) < reproj(rng.standard_normal((600, D_pix))*X.std()) * 0.25
# 生成图各维方差量级应与真实数据相近（不是全 0、不是爆炸）
ratio = x_out.var() / X.var()
print(f'生成图方差 / 真实数据方差 = {ratio:.2f}')
assert 0.3 < ratio < 3.0, '生成图的尺度应与真实数据同量级'
print('✅ 练习 4 通过：端到端 latent 扩散管线生成的图落在数据流形、尺度合理')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def forward_noise(z0, t, eps, abar):
    a = abar[t]
    return np.sqrt(a) * z0 + np.sqrt(1 - a) * eps

In [ ]:
# 练习 2 参考答案
def ddpm_step(z_t, t, eps_hat, alphas, abar, betas, noise):
    a_t = alphas[t]; ab_t = abar[t]
    mean = (z_t - (1 - a_t) / np.sqrt(1 - ab_t) * eps_hat) / np.sqrt(a_t)
    if t > 0:
        return mean + np.sqrt(betas[t]) * noise
    return mean

In [ ]:
# 练习 3 参考答案
def make_linear_vae(X, k):
    mean = X.mean(0)
    _, _, Vt = np.linalg.svd(X - mean, full_matrices=False)
    Wk = Vt[:k]
    def encode(x): return (x - mean) @ Wk.T
    def decode(z): return z @ Wk + mean
    return encode, decode, X.shape[1] / k

In [ ]:
# 练习 4 参考答案
def latent_pipeline(n_samples, seed=7):
    z_gen = ddpm_sample(n_samples, abar, betas, optimal_eps, seed=seed)
    return decode_from_diffusion(z_gen)

---
## 🧪 真实数据胶囊：Stable Diffusion 的算力账

用**真实 SD 的配置**算 latent 扩散到底省了多少。下面是公开的 SD/SDXL/SD3 配置（分辨率、下采样因子 f、latent 通道数 c）。

算每个模型「像素维度 vs latent 维度」的比值——这就是扩散每步省下的计算量级。

In [ ]:
# 真实模型配置（公开资料）
SD_CONFIGS = {
    'SD 1.5':  dict(res=512,  f=8,  c=4),
    'SDXL':    dict(res=1024, f=8,  c=4),
    'SD3':     dict(res=1024, f=8,  c=16),   # 16 通道 VAE
}
print(f"{'模型':<10}{'分辨率':>8}{'像素维度':>12}{'latent维度':>12}{'压缩比':>8}")
for name, cfg in SD_CONFIGS.items():
    pix = cfg['res']**2 * 3
    lat = (cfg['res']//cfg['f'])**2 * cfg['c']
    ratio = pix / lat
    print(f"{name:<10}{cfg['res']:>8}{pix:>12,}{lat:>12,}{ratio:>7.0f}x")
# SD1.5: 3·8²/4 = 48x ；SD3 通道翻到16 -> 压缩比降为 12x（换更好重建）
print('\n观察：SD1.5/SDXL 压缩 48x；SD3 用 16 通道 VAE 压缩比降到 12x —— 牺牲一点算力换重建质量')

**🧪 胶囊练习**：实现 `compression_ratio(res, f, c)` 返回像素维度与 latent 维度之比 `3·f²/c`，并算出：若要把 SD3 的压缩比拉回 SD1.5 的 48x，在 `f=8` 下 latent 通道数 `c` 该是多少？

In [ ]:
def compression_ratio(res, f, c):
    # TODO: 返回 (res²·3) / ((res//f)²·c)，等价于 3·f²/c
    raise NotImplementedError

def channels_for_ratio(target_ratio, f):
    # TODO: 由 3·f²/c = target_ratio 解出 c = 3·f²/target_ratio
    raise NotImplementedError

In [ ]:
# 自测
assert abs(compression_ratio(512, 8, 4) - 48.0) < 1e-6
assert abs(compression_ratio(1024, 8, 16) - 12.0) < 1e-6
# 压缩比与分辨率无关（只看 f 和 c）
assert abs(compression_ratio(512, 8, 4) - compression_ratio(2048, 8, 4)) < 1e-6
c_need = channels_for_ratio(48.0, 8)
assert abs(c_need - 4.0) < 1e-6, 'f=8 下要 48x 压缩比，需 4 通道'
print(f'要在 f=8 下达到 48x 压缩比，latent 需要 {c_need:.0f} 通道（正是 SD1.5 的选择）')
print('✅ 胶囊练习通过：理解 f 与通道数如何决定压缩比')

In [ ]:
# 📖 胶囊参考答案
def compression_ratio(res, f, c):
    return (res**2 * 3) / ((res // f)**2 * c)
def channels_for_ratio(target_ratio, f):
    return 3 * f**2 / target_ratio

---
### 小结
- **latent 扩散 = 先 VAE 压到低维 latent，在 latent 上做和像素扩散一模一样的加噪/去噪，最后解码一次**。
- **两阶段分工**：VAE 管感知压缩（去人眼冗余、确定性、训一次），扩散管语义生成（建模分布、迭代、随机）。
- **省算力**：每步约省 `3f²/c` 倍（SD 是 48x），采样多步把节省放大；但质量上限被 VAE 重建卡死。
- **工程细节**：latent 缩放（SD 的 0.18215）让方差≈1，匹配扩散噪声假设，不做会发散。
- **SD 架构 = VAE + 去噪骨干 + 文本编码器**；后续模块只是替换其中的方框（骨干→DiT、目标→flow、采样→少步）。

下一站：**模块 02 · Diffusion Transformer** —— 把去噪骨干从 U-Net 换成 Transformer，看 SD3/Flux/SoRA 的共同大脑。